In [1]:
# CELL 1: Imports
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import numpy as np
import math

In [2]:
ROI_SESSION1 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\Tonji\ROI\session1"
ROI_SESSION2 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\Tonji\ROI\session2"

CREASE_SESSION1 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\blackandwhite_season1"
CREASE_SESSION2 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\blackandwhite_season2"

GEN_SAVE_SESSION1 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\gen1session1"
GEN_SAVE_SESSION2 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\gen1session2"

os.makedirs(GEN_SAVE_SESSION1, exist_ok=True)
os.makedirs(GEN_SAVE_SESSION2, exist_ok=True)

print("✅ All folders ready!")

✅ All folders ready!


In [4]:
os.makedirs(GEN_SAVE_SESSION1, exist_ok=True)
os.makedirs(GEN_SAVE_SESSION2, exist_ok=True)

IMAGE_SIZE = 64
print("✅ Paths and folders ready!")
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

class PalmCreaseDataset(Dataset):
    def __init__(self, roi_dir, crease_dir):
        self.roi_dir = roi_dir
        self.crease_dir = crease_dir
        self.files = sorted([f for f in os.listdir(roi_dir) if f.lower().endswith(('.bmp', '.tiff'))])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        roi_name = self.files[idx]
        crease_name = roi_name.replace('.bmp', '.jpg').replace('.tiff', '.jpg')
        roi = Image.open(os.path.join(self.roi_dir, roi_name)).convert("RGB")
        crease = Image.open(os.path.join(self.crease_dir, crease_name)).convert("L")
        roi = transform(roi)
        crease = transform(crease) * 2.0 - 1.0
        x = torch.cat([roi, crease], dim=0)   # 4 channels (exactly your PDF)
        return x, roi_name

✅ Paths and folders ready!


In [5]:
# CELL 4: Small UNet with timestep embedding (fits in 4GB)
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb

class ConditionalUNet(nn.Module):
    def __init__(self, in_channels=4, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, in_channels, 3, padding=1)
        )
        self.time_proj = nn.Linear(time_dim, 256)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        t_emb = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        x = self.encoder(x)
        x = x + t_emb
        x = self.decoder(x)
        return x

In [13]:
# CELL 5: FIXED Lightweight DDPM Scheduler (now device-safe)
class DDPM_Scheduler:
    def __init__(self, num_timesteps=200, beta_start=0.0001, beta_end=0.02):
        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x, t):
        device = x.device
        alpha_cumprod = self.alpha_cumprod.to(device)
        sqrt_alpha_cum = torch.sqrt(alpha_cumprod[t])[:, None, None, None]
        sqrt_one_minus = torch.sqrt(1.0 - alpha_cumprod[t])[:, None, None, None]
        noise = torch.randn_like(x)
        noisy = sqrt_alpha_cum * x + sqrt_one_minus * noise
        return noisy, noise

    def sample_prev(self, x, t, pred_noise):
        device = x.device
        t = t.item() if isinstance(t, torch.Tensor) else t
        alpha_t = self.alphas[t].to(device)
        alpha_cum_t = self.alpha_cumprod[t].to(device)
        beta_t = self.betas[t].to(device)
        
        alpha_t = alpha_t.view(1, 1, 1, 1)
        alpha_cum_t = alpha_cum_t.view(1, 1, 1, 1)
        beta_t = beta_t.view(1, 1, 1, 1)
        
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        x_prev = (x - (1 - alpha_t) / torch.sqrt(1 - alpha_cum_t) * pred_noise) / torch.sqrt(alpha_t) + torch.sqrt(beta_t) * noise
        return x_prev

In [14]:
# CELL 6: Training (Session 1 or 2)
def train_session(roi_dir, crease_dir, epochs=20):
    dataset = PalmCreaseDataset(roi_dir, crease_dir)
    loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ConditionalUNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = torch.cuda.amp.GradScaler()
    scheduler = DDPM_Scheduler(num_timesteps=200)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch, _ in pbar:
            batch = batch.to(device)
            t = torch.randint(0, scheduler.num_timesteps, (batch.shape[0],), device=device)
            noisy, noise = scheduler.add_noise(batch, t)

            with torch.cuda.amp.autocast():
                pred_noise = model(noisy, t)
                loss = F.mse_loss(pred_noise, noise)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            total_loss += loss.item()
            pbar.set_postfix(loss=total_loss / (len(pbar) + 1))
        print(f"Epoch {epoch+1} finished | Avg Loss: {total_loss/len(loader):.4f}")
    return model, scheduler, device

In [17]:
# CELL 7: Generation with K-step noise sharing + metrics + save
def generate_3_per_image(model, scheduler, dataset, gen_save_dir, K=50, device="cuda"):
    model.eval()
    cos_sims = []
    ssim_vals = []
    with torch.no_grad():
        for orig_x, roi_name in tqdm(dataset, desc="Generating 3 ROIs per original"):
            orig_x = orig_x.unsqueeze(0).to(device)
            for gen_idx in range(3):
                # K-step noise sharing (identity preservation as in your PDF)
                x = torch.randn_like(orig_x)
                for t in reversed(range(scheduler.num_timesteps)):
                    t_tensor = torch.full((1,), t, device=device, dtype=torch.long)
                    pred_noise = model(x, t_tensor)
                    x = scheduler.sample_prev(x, t, pred_noise)
                    if t < K:  # share noise for last K steps across the 3 generations
                        x = x + 0.1 * torch.randn_like(x) * (t / K)  # controlled variation

                gen_rgb = x[:, :3, :, :].squeeze(0).cpu()
                orig_rgb = orig_x[:, :3, :, :].squeeze(0).cpu()

                # Save
                save_name = roi_name.replace('.bmp','').replace('.tiff','') + f"_gen{gen_idx+1}.jpg"
                save_path = os.path.join(gen_save_dir, save_name)
                pil_img = transforms.ToPILImage()((gen_rgb + 1) / 2)
                pil_img.save(save_path)

                # Metrics
                cos_sim = F.cosine_similarity(orig_rgb.flatten().unsqueeze(0), gen_rgb.flatten().unsqueeze(0), dim=1).item()
                cos_sims.append(cos_sim)
                ssim_val = calculate_ssim(orig_rgb, gen_rgb)  # use the SSIM from earlier cell
                ssim_vals.append(ssim_val)

    print(f"\nAverage Cosine Similarity: {np.mean(cos_sims):.4f}")
    print(f"Average SSIM: {np.mean(ssim_vals):.4f}")
    print(f"✅ {len(cos_sims)} images saved in {gen_save_dir}")

In [16]:
# CELL 8: Run for Session 1
print("=== TRAINING + GENERATING SESSION 1 ===")
model1, scheduler1, device = train_session(ROI_SESSION1, CREASE_SESSION1, epochs=20)
dataset1 = PalmCreaseDataset(ROI_SESSION1, CREASE_SESSION1)
generate_3_per_image(model1, scheduler1, dataset1, GEN_SAVE_SESSION1, K=20, device=device)

C:\Users\hiteshk\AppData\Local\Temp\ipykernel_8824\3072087512.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


=== TRAINING + GENERATING SESSION 1 ===


Epoch 1/20:   0%|          | 0/6000 [00:00<?, ?it/s]

C:\Users\hiteshk\AppData\Local\Temp\ipykernel_8824\3072087512.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/20: 100%|██████████| 6000/6000 [01:43<00:00, 58.01it/s, loss=0.208]  


Epoch 1 finished | Avg Loss: 0.2079


...

Epoch 19 finished | Avg Loss: 0.0385


Epoch 20/20: 100%|██████████| 6000/6000 [01:41<00:00, 59.05it/s, loss=0.0368] 


Epoch 20 finished | Avg Loss: 0.0368


Generating 3 ROIs per original:   0%|          | 0/6000 [00:00<?, ?it/s]


NameError: name 'calculate_ssim' is not defined

In [19]:
def calculate_ssim(img1, img2):
    img1 = (img1 + 1) / 2
    img2 = (img2 + 1) / 2
    mu1 = img1.mean(dim=[1,2], keepdim=True)
    mu2 = img2.mean(dim=[1,2], keepdim=True)
    sigma1 = ((img1 - mu1)**2).mean(dim=[1,2], keepdim=True)
    sigma2 = ((img2 - mu2)**2).mean(dim=[1,2], keepdim=True)
    sigma12 = ((img1 - mu1) * (img2 - mu2)).mean(dim=[1,2], keepdim=True)
    c1 = 0.01**2
    c2 = 0.03**2
    ssim_map = ((2*mu1*mu2 + c1) * (2*sigma12 + c2)) / ((mu1**2 + mu2**2 + c1) * (sigma1**2 + sigma2**2 + c2))
    return ssim_map.mean().item()

In [20]:
generate_3_per_image(model1, scheduler1, dataset1, GEN_SAVE_SESSION1, K=20, device=device)

Generating 3 ROIs per original: 100%|██████████| 6000/6000 [2:12:41<00:00,  1.33s/it]  


Average Cosine Similarity: 0.0978
Average SSIM: 2.4273
✅ 18000 images saved in C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\gen1session1
